# The Shape of the Gita: semantic map & community characterisation

A deep dive that pairs two questions into one picture:

1. **What does the Gita look like as a map?** Every verse already carries a
   pinned 768-dimensional embedding of its English translation. Projecting all
   701 into 2D with **t-SNE** lays the whole text out as a landscape, where
   verses that mean similar things sit near each other.
2. **What is each region *about*?** GDS **Louvain** communities (from the
   `SIMILAR_TO` network) colour the map; then each community is characterised by
   its most distinctive themes and concepts, and an exemplar verse nearest its
   centroid.

Read-only: this notebook never writes to the graph. Every section ends in inline
`assert`s. Figures are exported to `exports/` (gitignored).

**Tooling note:** the 2D projection uses scikit-learn **t-SNE** (already a
dependency) rather than UMAP. UMAP pulls in `numba`, which lacks wheels on this
project's Python. t-SNE needs no extra install and produces the same kind of map.

**Prerequisites:** the graph built by `gita_kg.ipynb` (embeddings + `SIMILAR_TO`
present), the GDS plugin, and `.env`.

## 1. Setup & connect

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from graphdatascience import GraphDataScience
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))

import gita_kg as gk

EXPORTS = PKG / "exports"
EXPORTS.mkdir(exist_ok=True)

/Users/Akhilesh.Koul/Documents/GitHub/CodePlayground/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(PKG / ".env", override=True)
cfg = gk.load_config()
gds = GraphDataScience(cfg.uri, auth=(cfg.user, cfg.password), database=cfg.database)
gds.set_show_progress(False)
print("GDS server:", gds.version(), "| database:", cfg.database)

GDS server: 2026.7.0 | database: neo4j


In [3]:
def cypher(query: str, **params) -> pd.DataFrame:
    return gds.run_cypher(query, params)


def drop_if_exists(name: str) -> None:
    if gds.graph.exists(name)["exists"]:
        gds.graph.drop(name)


def export(fig: go.Figure, filename: str) -> None:
    out = EXPORTS / filename
    fig.write_html(out, include_plotlyjs="cdn")
    print("wrote", out.relative_to(ROOT))


PALETTE = px.colors.qualitative.Bold

## 2. Pull verses, embeddings & communities

In [4]:
verse_df = cypher(
    "MATCH (v:Verse) RETURN id(v) AS nodeId, v.id AS id, v.chapter AS chapter, "
    "v.verse AS verse, v.translation AS translation, v.embedding AS emb "
    "ORDER BY v.chapter, v.verse"
)
X = np.array(verse_df["emb"].tolist(), dtype=np.float32)
verse_df = verse_df.drop(columns=["emb"])

# Correctness: full coverage, right shape, embeddings unit-normalised.
assert X.shape == (701, 768), X.shape
assert np.allclose(np.linalg.norm(X, axis=1), 1.0, atol=1e-4)
print("embedding matrix:", X.shape)

embedding matrix: (701, 768)


In [5]:
# GDS Louvain communities on the SIMILAR_TO network colour the map.
SIM = "verse_similarity_map"
drop_if_exists(SIM)
G, _ = gds.graph.project(
    SIM, "Verse",
    {"SIMILAR_TO": {"orientation": "UNDIRECTED", "properties": "score"}},
)
assert G.node_count() == 701, G.node_count()
lv = gds.louvain.stream(G, relationshipWeightProperty="score")
comm_map = lv.set_index("nodeId")["communityId"]
verse_df["community"] = verse_df["nodeId"].map(comm_map)
assert verse_df["community"].notna().all()

sizes = verse_df["community"].value_counts()
TOP_N = 8
top_comms = sizes.head(TOP_N).index.tolist()
# Colour the TOP_N largest communities; everything else is "other" (grey).
verse_df["comm_label"] = np.where(
    verse_df["community"].isin(top_comms),
    "C" + verse_df["community"].astype(str), "other",
)
print(f"{sizes.size} communities; colouring the {TOP_N} largest, "
      f"{int(sizes.head(TOP_N).sum())}/{len(verse_df)} verses")

47 communities; colouring the 8 largest, 474/701 verses


In [6]:
# Dominant theme per verse (for hover + later labelling).
vt = cypher(
    "MATCH (v:Verse)-[m:MENTIONS_THEME]->(t:Theme) "
    "RETURN v.id AS id, t.name AS theme, m.weight AS weight"
)
dominant = (
    vt.sort_values("weight", ascending=False)
    .drop_duplicates("id")[["id", "theme"]]
    .rename(columns={"theme": "top_theme"})
)
verse_df = verse_df.merge(dominant, on="id", how="left")
verse_df["top_theme"] = verse_df["top_theme"].fillna("none")


def snippet(text: str, n: int = 90) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n].rsplit(" ", 1)[0] + "…"


verse_df["hover"] = (
    verse_df["id"] + " · ch" + verse_df["chapter"].astype(str)
    + " · " + verse_df["top_theme"] + "<br>"
    + verse_df["translation"].map(snippet)
)

## 3. The semantic map (t-SNE)

In [7]:
coords = TSNE(
    n_components=2, init="pca", perplexity=30, random_state=42, metric="cosine",
).fit_transform(X)
assert coords.shape == (701, 2), coords.shape
verse_df["x"] = coords[:, 0]
verse_df["y"] = coords[:, 1]
print("t-SNE projection ready:", coords.shape)

t-SNE projection ready: (701, 2)


In [8]:
# Figure 1: the map coloured by Louvain community.
order = ["C" + str(c) for c in top_comms] + ["other"]
cmap = {lab: PALETTE[i % len(PALETTE)] for i, lab in enumerate(order[:-1])}
cmap["other"] = "#d9d9d9"
fig_comm = px.scatter(
    verse_df, x="x", y="y", color="comm_label",
    category_orders={"comm_label": order}, color_discrete_map=cmap,
    custom_data=["hover"],
    title="The shape of the Gita: 701 verses, coloured by semantic community",
)
fig_comm.update_traces(
    marker=dict(size=6, opacity=0.85, line=dict(width=0)),
    hovertemplate="%{customdata[0]}<extra></extra>",
)
fig_comm.update_layout(
    height=680, legend_title="community",
    xaxis=dict(visible=False), yaxis=dict(visible=False),
)
export(fig_comm, "map_verse_communities.html")
fig_comm

wrote gita-knowledge-graph/exports/map_verse_communities.html


In [9]:
# Figure 2: the same map coloured by chapter (does reading order = geography?)
fig_chap = px.scatter(
    verse_df, x="x", y="y", color="chapter",
    color_continuous_scale="Turbo", custom_data=["hover"],
    title="The same map, coloured by chapter (1 → 18)",
)
fig_chap.update_traces(
    marker=dict(size=6, opacity=0.85),
    hovertemplate="%{customdata[0]}<extra></extra>",
)
fig_chap.update_layout(
    height=680, xaxis=dict(visible=False), yaxis=dict(visible=False),
)
export(fig_chap, "map_verse_chapters.html")
fig_chap

wrote gita-knowledge-graph/exports/map_verse_chapters.html


## 4. What is each community about?

Raw theme weight is dominated by ubiquitous themes (karma is everywhere), so a
community's *character* is better shown by **lift**, how much more a theme
appears in that community than in the text overall. Below: the share heatmap for
readability, plus a lift-based distinctive label per community.

In [10]:
# Community x theme weight -> per-community share, and global share for lift.
ct = cypher(
    "MATCH (v:Verse)-[m:MENTIONS_THEME]->(t:Theme) "
    "RETURN id(v) AS nodeId, t.name AS theme, m.weight AS weight"
)
ct["community"] = ct["nodeId"].map(comm_map)
comm_theme = ct.groupby(["community", "theme"])["weight"].sum().unstack(fill_value=0.0)
share = comm_theme.div(comm_theme.sum(axis=1), axis=0)
# Each community's theme shares sum to 1.
assert np.allclose(share.sum(axis=1).values, 1.0)

global_share = comm_theme.sum(axis=0) / comm_theme.sum().sum()
lift = share.div(global_share, axis=1)
print("theme x community share computed for", share.shape[0], "communities")

theme x community share computed for 31 communities


In [11]:
# Figure 3: theme-share heatmap for the coloured communities.
heat = share.loc[[c for c in top_comms if c in share.index]]
heat.index = ["C" + str(c) + f" (n={sizes[c]})" for c in heat.index]
fig_sig = px.imshow(
    heat.T, aspect="auto", color_continuous_scale="Viridis",
    labels=dict(x="community", y="theme", color="share"),
    title="Theme signature of each community (share of theme weight)",
)
fig_sig.update_layout(height=560)
export(fig_sig, "map_community_theme_signature.html")
fig_sig

wrote gita-knowledge-graph/exports/map_community_theme_signature.html


In [12]:
# Distinctive label per community (top themes by lift, with support floor),
# top concept by share, and the exemplar verse nearest the centroid.
concept = cypher(
    "MATCH (v:Verse)-[m:EXPRESSES_CONCEPT]->(c:Concept) "
    "RETURN id(v) AS nodeId, c.name AS concept, m.weight AS weight"
)
concept["community"] = concept["nodeId"].map(comm_map)
comm_concept = concept.groupby(["community", "concept"])["weight"].sum().unstack(fill_value=0.0)
comm_concept_share = comm_concept.div(comm_concept.sum(axis=1), axis=0)

nodeid_to_row = {nid: i for i, nid in enumerate(verse_df["nodeId"])}
rows = []
for c in top_comms:
    members = verse_df.index[verse_df["community"] == c]
    idx = [nodeid_to_row[n] for n in verse_df.loc[members, "nodeId"]]
    centroid = X[idx].mean(axis=0)
    centroid /= np.linalg.norm(centroid) or 1.0
    sims = X[idx] @ centroid
    exemplar = verse_df.loc[members[int(np.argmax(sims))]]
    # exemplar must belong to this community.
    assert exemplar["community"] == c

    supported = share.loc[c][share.loc[c] >= 0.03].index
    distinctive = lift.loc[c, supported].sort_values(ascending=False).head(3)
    top_by_share = share.loc[c].sort_values(ascending=False).head(2).index.tolist()
    top_concept = (
        comm_concept_share.loc[c].sort_values(ascending=False).index[0]
        if c in comm_concept_share.index else "none"
    )
    rows.append({
        "community": f"C{c}", "verses": int(sizes[c]),
        "distinctive_themes": ", ".join(distinctive.index),
        "top_themes_by_share": ", ".join(top_by_share),
        "top_concept": top_concept,
        "exemplar": exemplar["id"],
        "exemplar_text": snippet(exemplar["translation"], 80),
    })

summary = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 90)
print(summary.to_string(index=False))

community  verses                   distinctive_themes        top_themes_by_share top_concept exemplar                                                                   exemplar_text
     C368      90               bhakti, samsara, jnana              jnana, bhakti      buddhi     12.6 But to those who worship Me, renouncing all actions in Me, regarding Me as the…
     C351      86                bhakti, dharma, jnana              jnana, bhakti     brahman     7.29   Those who strive for liberation from old age and death, taking refuge in Me,…
     C116      73             karma, detachment, atman          karma, detachment       karma     3.30    Renouncing all actions in Me, with the mind centered on the Self, free from…
     C462      49     guna, sacrifice-austerity, jnana             jnana, samsara       atman     11.4  If Thou, O Lord, thinkest it possible for me to see it, do Thou, then, O Lord…
     C354      47                 atman, guna, samsara               atman, jnana    

In [13]:
# Figure 4: the map again, annotated with each community's distinctive theme.
fig_lab = go.Figure(fig_comm)
for c in top_comms:
    members = verse_df["community"] == c
    cx = verse_df.loc[members, "x"].mean()
    cy = verse_df.loc[members, "y"].mean()
    theme = summary.loc[summary["community"] == f"C{c}",
                        "distinctive_themes"].iloc[0].split(",")[0].strip()
    fig_lab.add_annotation(
        x=cx, y=cy, text=f"<b>{theme}</b>", showarrow=False,
        font=dict(size=13, color="#111111"),
        bgcolor="rgba(255,255,255,0.7)", borderpad=2,
    )
fig_lab.update_layout(title="Semantic map, annotated by each region's most distinctive theme")
export(fig_lab, "map_verse_annotated.html")
fig_lab

wrote gita-knowledge-graph/exports/map_verse_annotated.html


## 5. Cleanup

In [14]:
drop_if_exists(SIM)
gds.close()
print("dropped projection, closed GDS session.")
print("figures written to", EXPORTS.relative_to(ROOT))

dropped projection, closed GDS session.
figures written to gita-knowledge-graph/exports
